# Phase 1: Data Acquisition & Structural Quality Checks

**Business question this phase serves:** *what data do we actually have, and can we trust it?*
Every GLM/GAM coefficient in this project ultimately traces back to the rows we keep or drop here.
so this notebook earns its place before any modeling notebook.

**Dataset.** `freMTPL2freq` (one row per policy) and `freMTPL2sev` (one row per claim payment),
one accounting year of a French motor third-party-liability book, hosted on
[OpenML](https://www.openml.org/) (data ids `41214` and `41215`). Field-level definitions live in
[`docs/data_dictionary.md`](../docs/data_dictionary.md): this notebook is where every quirk quoted
in that dictionary was actually measured.

**What this notebook does, in order:**
1. Load the raw tables through the shared `src/data/load_data.py` loader (never re-implemented here).
2. Structural checks: duplicate policies, exposure range, claim-count range, missing values.
3. Reconcile the frequency table's `ClaimNb` against the severity table's claim payments.
   this is the single most important data-quality question in the whole dataset, because it decides
   what our frequency and severity *targets* actually mean.
4. Build one clean, policy-level table and persist it to `data/processed/`, ready for Phase 2 (EDA).

**A note on method, up front:** every number quoted in prose below is computed in the code cell right
above it: nothing here is asserted from memory of "what this dataset usually looks like." That
discipline is what makes a data-quality section trustworthy to a reviewer.

## 0. Setup

In [1]:
import sys
from pathlib import Path

import numpy as np
import pandas as pd

# Make the project's src/ package importable regardless of where Jupyter was
# launched from, by walking up from the current directory until we find a
# file we know only exists at the project root.
def find_project_root(marker: str = "requirements.txt") -> Path:
    here = Path.cwd().resolve()
    for candidate in [here, *here.parents]:
        if (candidate / marker).exists():
            return candidate
    raise FileNotFoundError(f"Could not locate project root (looked for '{marker}' upward from {here})")

PROJECT_ROOT = find_project_root()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

print("Project root found:", PROJECT_ROOT.name)

# Display options: this dataset has ~678k rows: keep summaries readable.
pd.set_option("display.max_columns", 20)
pd.set_option("display.float_format", lambda x: f"{x:,.4f}")

Project root found: COMPLETE-AUTO-PRICING-GLM_GAM


## 1. Load the raw tables

Both tables are fetched once from OpenML and cached locally as Parquet by `src/data/load_data.py`
(see that module's docstring for why this logic lives in `src/` and not here). If this is the first
run on this machine, the next cell downloads ~678k policy rows and ~27k claim rows (a few seconds);
every run after that reads from the local cache.

In [2]:
from src.data.load_data import load_freq_raw, load_sev_raw

freq = load_freq_raw()
sev = load_sev_raw()

print(f"freMTPL2freq: {freq.shape[0]:,} rows x {freq.shape[1]} columns")
print(f"freMTPL2sev : {sev.shape[0]:,} rows x {sev.shape[1]} columns")

freMTPL2freq: 678,013 rows x 12 columns
freMTPL2sev : 26,639 rows x 2 columns


In [3]:
freq.head()

,IDpol,ClaimNb,Exposure,Area,VehPower,VehAge,DrivAge,BonusMalus,VehBrand,VehGas,Density,Region
0,1,1,0.1000,D,5,0,55,50,B12,Regular,"1,217.0000",R82
1,3,1,0.7700,D,5,0,55,50,B12,Regular,"1,217.0000",R82
2,5,1,0.7500,B,6,2,52,50,B12,Diesel,54.0000,R22
3,10,1,0.0900,B,7,0,46,50,B12,Diesel,76.0000,R72
4,11,1,0.8400,B,7,0,46,50,B12,Diesel,76.0000,R72


In [4]:
sev.head()

,IDpol,ClaimAmount
0,1552,995.2000
1,1010996,"1,128.1200"
2,4024277,"1,851.1100"
3,4007252,"1,204.0000"
4,4046424,"1,204.0000"


In [5]:
freq.dtypes

IDpol            int64
ClaimNb          int64
Exposure       float64
Area          category
VehPower         int64
VehAge           int64
DrivAge          int64
BonusMalus       int64
VehBrand      category
VehGas        category
Density        float64
Region        category
dtype: object

## 2. Structural checks: `freMTPL2freq`

### 2.1 Duplicate policies

Some public redistributions of this dataset are known to contain duplicate `IDpol` rows. We check
directly rather than assume: and, whichever way it comes out, we say so precisely instead of citing
"a known issue with this dataset" without checking our own copy.

In [6]:
n_rows = len(freq)
n_unique_ids = freq["IDpol"].nunique()
n_full_dupes = freq.duplicated().sum()
n_id_dupes = freq["IDpol"].duplicated().sum()

print(f"Rows                     : {n_rows:,}")
print(f"Unique IDpol             : {n_unique_ids:,}")
print(f"Full-row duplicates      : {n_full_dupes:,}")
print(f"Rows with a duplicated IDpol (any) : {n_id_dupes:,}")

Rows                     : 678,013
Unique IDpol             : 678,013
Full-row duplicates      : 0
Rows with a duplicated IDpol (any) : 0


**Finding:** in this OpenML pull, `IDpol` is unique across all rows and there are zero full-row
duplicates. Whatever duplicate-row issue is reported for other redistributions of this dataset does
not apply to the copy this project is built on: no deduplication step is needed here, and none is
applied. (If this changes on a future re-download from OpenML, this cell will catch it immediately.
that's the point of checking programmatically instead of hard-coding a cleaning step.)

### 2.2 Exposure

`Exposure` is the fraction of the one-year observation window each policy was on risk (e.g. `0.5` =
six months). It is not a rating factor, it will be used as the GLM/GAM **offset** in Phase 4, but it
still needs to be sane: a policy cannot physically be on-risk for more than the observation window.

In [7]:
display(freq["Exposure"].describe())

n_exposure_over_1 = (freq["Exposure"] > 1).sum()
n_exposure_nonpositive = (freq["Exposure"] <= 0).sum()

print(f"\nPolicies with Exposure > 1 year : {n_exposure_over_1:,} ({n_exposure_over_1 / n_rows:.3%})")
print(f"Policies with Exposure <= 0     : {n_exposure_nonpositive:,}")
print(f"Max exposure observed           : {freq['Exposure'].max():.3f} years")

count   678,013.0000
mean          0.5288
std           0.3644
min           0.0027
25%           0.1800
50%           0.4900
75%           0.9900
max           2.0100
Name: Exposure, dtype: float64


Policies with Exposure > 1 year : 1,224 (0.181%)
Policies with Exposure <= 0     : 0
Max exposure observed           : 2.010 years


**Finding & decision:** a small minority of policies (well under 1%: see count above) show
exposure slightly above one year, up to just over two years at the extreme, most plausibly a
data-entry/period-boundary artifact rather than a real >1-year single observation window. No policy
has zero or negative exposure. **Decision:** exposure is capped at `1.0` when we build the modeling
table in Section 5: capping (not dropping) preserves these policies' rating-factor information while
removing the physically-impossible offset value.

### 2.3 Claim counts (`ClaimNb`)

This is the frequency target. Motor TPL frequency is a rare-event count: we expect a distribution
dominated by zeros, with a thin tail.

In [8]:
display(freq["ClaimNb"].describe())
print()
display(freq["ClaimNb"].value_counts().sort_index())

pct_zero = (freq["ClaimNb"] == 0).mean()
n_high = (freq["ClaimNb"] > 4).sum()
print(f"\nShare of policies with zero claims : {pct_zero:.2%}")
print(f"Policies with more than 4 claims   : {n_high:,}")
print(f"Max ClaimNb observed               : {freq['ClaimNb'].max()}")

count   678,013.0000
mean          0.0532
std           0.2401
min           0.0000
25%           0.0000
50%           0.0000
75%           0.0000
max          16.0000
Name: ClaimNb, dtype: float64

ClaimNb
0     643953
1      32178
2       1784
3         82
4          7
5          2
6          1
8          1
9          1
11         3
16         1
Name: count, dtype: int64


Share of policies with zero claims : 94.98%
Policies with more than 4 claims   : 9
Max ClaimNb observed               : 16


**Finding:** claim frequency behaves as expected for motor TPL: the large majority of policies
report zero claims, and the observed portfolio-average frequency is a few percent per year of
exposure (we compute the *exposure-weighted* frequency properly in Phase 2; the raw mean of `ClaimNb`
above is not exposure-adjusted and is only a first sanity check). A very small number of policies
report more than 4 claims in the year, up to a maximum in the double digits: plausible for a
large-fleet or professional policy rather than an obvious data error, but flagged here so the
decision to keep or cap it is made deliberately in Phase 3 feature preparation, not by accident.

### 2.4 Missing values

In [9]:
freq.isnull().sum().to_frame("n_missing")

,n_missing
IDpol,0
ClaimNb,0
Exposure,0
Area,0
VehPower,0
VehAge,0
DrivAge,0
BonusMalus,0
VehBrand,0
VehGas,0


**Finding:** no missing values anywhere in `freMTPL2freq`.

## 3. Structural checks: `freMTPL2sev`

`freMTPL2sev` has one row per claim **payment** (not per claim event, and not per policy): a single
claim can in principle generate more than one payment row.

In [10]:
display(sev["ClaimAmount"].describe())

n_nonpositive = (sev["ClaimAmount"] <= 0).sum()
print(f"\nClaimAmount <= 0 : {n_nonpositive:,}")
print(f"Missing values   :")
display(sev.isnull().sum())

count      26,639.0000
mean        2,278.5364
std        29,297.4813
min             1.0000
25%           686.8100
50%         1,172.0000
75%         1,228.0800
max     4,075,400.5600
Name: ClaimAmount, dtype: float64


ClaimAmount <= 0 : 0
Missing values   :


IDpol          0
ClaimAmount    0
dtype: int64

**Finding:** no missing or non-positive claim amounts. The distribution is, as expected for
motor TPL, heavily right-skewed: the median payment is roughly €1,200 but the maximum single payment
is in the millions of euros: a small number of very large bodily-injury-type claims sit far out in
the tail. This is exactly the shape that motivates using a **Gamma** (not Gaussian) GLM for severity
in Phase 6, and it is why a handful of extreme claims will get explicit attention there rather than
being allowed to dominate a mean-based fit unexamined.

## 4. Reconciling `ClaimNb` against `freMTPL2sev`: the key data-quality question

`ClaimNb` (frequency file) and "number of matching rows in `freMTPL2sev`" are two different measures
of "how many claims did this policy have," and **they do not have to agree**: `ClaimNb` reflects claims
*reported/reserved*, while `freMTPL2sev` reflects claims that generated an actual *payment* within
this extract's accounting period. Understanding exactly how and where they disagree is what lets us
choose defensible frequency and severity targets, instead of picking one arbitrarily.

In [11]:
# One row per policy, with the number of matching severity rows and their total amount.
sev_agg = (
    sev.groupby("IDpol")
    .agg(sev_claim_count=("ClaimAmount", "size"), sev_claim_amount=("ClaimAmount", "sum"))
    .reset_index()
)

merged = freq.merge(sev_agg, on="IDpol", how="left")
merged["sev_claim_count"] = merged["sev_claim_count"].fillna(0).astype(int)
merged["sev_claim_amount"] = merged["sev_claim_amount"].fillna(0.0)

n_mismatch = (merged["ClaimNb"] != merged["sev_claim_count"]).sum()
n_claimnb_pos_sev_zero = ((merged["ClaimNb"] > 0) & (merged["sev_claim_count"] == 0)).sum()
n_claimnb_zero_sev_pos = ((merged["ClaimNb"] == 0) & (merged["sev_claim_count"] > 0)).sum()
n_both_pos_diff = ((merged["ClaimNb"] > 0) & (merged["sev_claim_count"] > 0) & (merged["ClaimNb"] != merged["sev_claim_count"])).sum()

print(f"Policies where ClaimNb != matching severity-row count : {n_mismatch:,} ({n_mismatch / n_rows:.2%} of all policies)")
print(f"  -> ClaimNb > 0 but 0 severity rows (reported, no payment in this extract) : {n_claimnb_pos_sev_zero:,}")
print(f"  -> ClaimNb == 0 but severity rows exist (claim 'from nowhere')           : {n_claimnb_zero_sev_pos:,}")
print(f"  -> both > 0 but counts differ (partial-payment / multi-row claims)        : {n_both_pos_diff:,}")

n_orphan_claims = sev.loc[~sev["IDpol"].isin(freq["IDpol"]), "IDpol"].nunique()
print(f"\nIDpol present in freMTPL2sev with NO matching policy in freMTPL2freq : {n_orphan_claims}")

Policies where ClaimNb != matching severity-row count : 9,117 (1.34% of all policies)
  -> ClaimNb > 0 but 0 severity rows (reported, no payment in this extract) : 9,116
  -> ClaimNb == 0 but severity rows exist (claim 'from nowhere')           : 0
  -> both > 0 but counts differ (partial-payment / multi-row claims)        : 1

IDpol present in freMTPL2sev with NO matching policy in freMTPL2freq : 6


**Findings:**
- About 1.3% of policies show a mismatch, and it is almost entirely one-directional: `ClaimNb > 0`
  with zero matching payment rows. No policy shows the opposite pattern (`ClaimNb == 0` with
  payments recorded).
- A single policy has payment rows that don't match its reported count even though both are positive.
- A handful of `IDpol` values in the severity file don't exist in the frequency file at all: claims
  with no policy record to attach risk factors to.

Before deciding how to treat the unpaid claims, the next cell checks *where* they sit in the file.

In [12]:
from src.data.load_data import find_claims_only_block_end

block_end = find_claims_only_block_end(freq)
in_block = merged["IDpol"] <= block_end
unpaid = (merged["ClaimNb"] > 0) & (merged["sev_claim_count"] == 0)

for label, mask in [("IDpol <= block end", in_block), ("IDpol >  block end", ~in_block)]:
    part = merged[mask]
    claims = part["ClaimNb"].sum()
    paid = part["sev_claim_count"].sum()
    print(
        f"{label}: {len(part):>7,} policies | share with a claim {(part['ClaimNb'] > 0).mean():6.1%} | "
        f"claim frequency {claims / part['Exposure'].clip(upper=1).sum():6.1%} | "
        f"claims with a payment {paid / claims:6.1%}"
    )
print(f"\nLast IDpol of the leading block : {block_end:,}")
print(f"Unpaid-claim policies inside the block  : {(unpaid & in_block).sum():,}")
print(f"Unpaid-claim policies outside the block : {(unpaid & ~in_block).sum():,}")
print(f"Exposure share of the block             : {merged.loc[in_block, 'Exposure'].clip(upper=1).sum() / merged['Exposure'].clip(upper=1).sum():.2%}")

IDpol <= block end:   9,387 policies | share with a claim 100.0% | claim frequency 207.1% | claims with a payment   3.1%
IDpol >  block end: 668,626 policies | share with a claim   3.7% | claim frequency   7.4% | claims with a payment  99.9%

Last IDpol of the leading block : 24,951
Unpaid-claim policies inside the block  : 9,094
Unpaid-claim policies outside the block : 22
Exposure share of the block             : 1.34%


**Finding: the unpaid claims are not spread across the portfolio, they sit in one block.** Every
policy at the start of the file, up to the IDpol printed above, has at least one reported claim:
there is no claim-free policy in that range at all, the implied claim frequency is above 100%, and
only about 3% of those claims have a payment in `freMTPL2sev`. Outside the block, virtually every
reported claim has a payment. This range is a claims file appended to the portfolio, not a sample of
the insured population.

**Modeling decisions (applied when building the processed table below):**
- **The leading claims-only block is excluded.** Kept, it would inflate the modeled claim frequency
  (its claims enter the frequency model) without adding the matching cost (its payments are missing),
  so the technical premium would no longer reconcile with paid claims. It also distorts rating
  relativities, because its mix of risk factors differs from the rest of the book (for example two
  thirds Regular fuel, against about half elsewhere). The boundary is detected from the data by
  `find_claims_only_block_end`, not hard-coded.
- **`ClaimNb` from the frequency file is the frequency target** for every remaining policy (including
  all zero-claim ones).
- **Severity is built from `freMTPL2sev` payments, aggregated to policy level**, and is only used for
  policies with at least one positive payment.
- **Orphan claims (no matching policy) are dropped**: there is no way to attach risk-factor
  information to them.

## 5. Build the clean, policy-level modeling table

Everything decided above gets applied here, once, producing the single table every later notebook
will read from `data/processed/`.

In [13]:
from src.data.load_data import drop_claims_only_block

# --- Exclude the leading claims-only block (Section 4) ---
policy = drop_claims_only_block(freq, block_end)
n_block = n_rows - len(policy)

# --- Exposure: cap at 1.0 (Section 2.2) ---
policy["Exposure_raw"] = policy["Exposure"]
policy["Exposure"] = policy["Exposure"].clip(upper=1.0)
n_capped = (policy["Exposure_raw"] > 1.0).sum()

# --- Attach severity aggregates (Section 4) ---
policy = policy.merge(sev_agg, on="IDpol", how="left")
policy["sev_claim_count"] = policy["sev_claim_count"].fillna(0).astype(int)
policy["sev_claim_amount"] = policy["sev_claim_amount"].fillna(0.0)

# --- Frequency target: ClaimNb, as decided in Section 4 ---
policy["ClaimNb_freq_target"] = policy["ClaimNb"]

# --- Severity target: average paid severity per policy, defined only where a payment exists ---
# (NaN, not 0, for policies with no payment: explicit "not applicable", not a fake zero claim cost)
policy["AvgClaimAmount"] = np.where(
    policy["sev_claim_count"] > 0,
    policy["sev_claim_amount"] / policy["sev_claim_count"],
    np.nan,
)
policy["HasPaidClaim"] = policy["sev_claim_count"] > 0

print(f"Policies excluded (claims-only block) : {n_block:,}")
print(f"Exposure values capped at 1.0        : {n_capped:,}")
print(f"Policies with >=1 paid claim (severity-model rows) : {policy['HasPaidClaim'].sum():,}")
print(f"Final policy-level table shape       : {policy.shape}")
policy.head()

Policies excluded (claims-only block) : 9,387
Exposure values capped at 1.0        : 1,224
Policies with >=1 paid claim (severity-model rows) : 24,651
Final policy-level table shape       : (668626, 18)


,IDpol,ClaimNb,Exposure,Area,VehPower,VehAge,DrivAge,BonusMalus,VehBrand,VehGas,Density,Region,Exposure_raw,sev_claim_count,sev_claim_amount,ClaimNb_freq_target,AvgClaimAmount,HasPaidClaim
0,24952,0,0.0055,A,6,15,50,50,B2,Diesel,29.0000,R24,0.0055,0,0.0000,0,NaN,False
1,24953,0,0.6700,A,6,15,50,50,B2,Diesel,29.0000,R24,0.6700,0,0.0000,0,NaN,False
2,24955,0,0.8900,A,6,8,30,68,B1,Diesel,48.0000,R53,0.8900,0,0.0000,0,NaN,False
3,24956,0,0.1000,A,6,8,30,57,B1,Diesel,48.0000,R53,0.1000,0,0.0000,0,NaN,False
4,24958,0,0.2000,B,7,4,44,50,B14,Diesel,56.0000,R24,0.2000,0,0.0000,0,NaN,False


## 6. Final validation and persistence

In [14]:
# Sanity re-check on the table we're about to save: no negative exposures/counts, no leftover
# infinities, and the only rows removed are the claims-only block (Section 4). The severity training
# set, built in Phase 6, filters further down to HasPaidClaim == True.
assert len(policy) == n_rows - n_block, "Row count changed unexpectedly while building the policy table"
assert (policy["IDpol"] > block_end).all(), "Claims-only block was not fully excluded"
assert (policy["Exposure"] <= 1.0).all(), "Exposure cap did not apply everywhere"
assert (policy["ClaimNb_freq_target"] >= 0).all(), "Negative claim count found"
assert np.isfinite(policy["sev_claim_amount"]).all(), "Non-finite severity amount found"

processed_dir = PROJECT_ROOT / "data" / "processed"
processed_dir.mkdir(parents=True, exist_ok=True)
out_path = processed_dir / "policy_level_clean.parquet"
policy.to_parquet(out_path, index=False)

print(f"Saved: {out_path.relative_to(PROJECT_ROOT).as_posix()}")
print(f"Rows : {len(policy):,}  |  Columns: {list(policy.columns)}")

Saved: data/processed/policy_level_clean.parquet
Rows : 668,626  |  Columns: ['IDpol', 'ClaimNb', 'Exposure', 'Area', 'VehPower', 'VehAge', 'DrivAge', 'BonusMalus', 'VehBrand', 'VehGas', 'Density', 'Region', 'Exposure_raw', 'sev_claim_count', 'sev_claim_amount', 'ClaimNb_freq_target', 'AvgClaimAmount', 'HasPaidClaim']


## Phase 1 summary

| Check | Result | Action taken |
|---|---|---|
| Duplicate `IDpol` | none found | none needed |
| Missing values (either table) | none found | none needed |
| `Exposure` > 1 year | 1,224 policies | capped at 1.0 |
| `Exposure` <= 0 | none found | none needed |
| `ClaimAmount` <= 0 | none found | none needed |
| `ClaimNb` vs severity-row count mismatch | 9,117 policies (~1.3%), 9,094 of them in one block | see next row |
| Leading claims-only block (IDpol <= 24,951) | 9,387 policies, all with a claim, 3.1% of claims paid, 1.34% of exposure | excluded: it is a claims extract, not part of the insured population |
| Orphan claims (no matching policy) | 6 `IDpol` | dropped (no risk factors to attach) |

**Output:** [`data/processed/policy_level_clean.parquet`](../data/processed/policy_level_clean.parquet):
one row per policy, 668,626 rows, ready for exposure-weighted exploratory analysis in
**Phase 2** ([`notebooks/02_exploratory_data_analysis.ipynb`](02_exploratory_data_analysis.ipynb)).

Every quirk and count documented in [`docs/data_dictionary.md`](../docs/data_dictionary.md) is
reproduced by this notebook.